# Pre-trained Encoder + JacobianODE — Lorenz (Loop Closure Sweep)

This notebook sweeps over **loop closure weight** values for the JacobianODE
trained on a pre-trained encoder's latent space.

**Pipeline:**
1. Load pre-trained encoder from W&B checkpoint
2. Generate data once (shared across all sweep runs)
3. For each `loop_closure_weight`, train a fresh JacobianODE (via SLURM or local loop)
4. Collect results from W&B, apply physics-informed model selection (C1/C2/C3)
5. Load the best model for downstream use

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import matplotlib.pyplot as plt
import numpy as np
from omegaconf import OmegaConf
import os
import torch
import wandb
from tqdm.auto import tqdm

from JacobianODE.jacobians import (
    load_config,
    initialize_config,
    seed_everything,
    make_trajectories,
    postprocess_data,
    create_dataloaders,
    train_model,
    load_run,
    load_checkpoint,
    select_best_model,
    DiagnosticMetrics,
)
from JacobianODE.encoder_only.pretrained import load_pretrained_encoder
from JacobianODE.models.latent_jacobian import LitLatentJacobianODE
from JacobianODE.jacobians.tuning import select_from_wandb_runs
from hydra.utils import instantiate

torch.set_float32_matmul_precision('high')

## 1. Settings

In [ ]:
# ----------------------------------------------------------------
# Paths and W&B settings
# ----------------------------------------------------------------
SAVE_DIR = "/orcd/data/ekmiller/001/eisenaj/JacobianODE/lightning/pretrained_jac_runs"
WANDB_ENTITY = "JacobianODE"
WANDB_PROJECT = None  # Auto-generated below

# ----------------------------------------------------------------
# Pre-trained encoder source (EDIT THESE)
# ----------------------------------------------------------------
ENCODER_PROJECT = "JacobianODE/<YOUR_ENCODER_PROJECT>"
ENCODER_RUN_ID = "<YOUR_RUN_ID>"

In [ ]:
# ----------------------------------------------------------------
# Sweep parameters
# ----------------------------------------------------------------
LAMBDA_LOOP_VALUES = [0.0, 0.001, 0.01, 0.1, 1.0]

FREEZE_ENCODER = True

In [ ]:
# ----------------------------------------------------------------
# Load pre-trained encoder
# ----------------------------------------------------------------
adapter, encoder_cfg, encoder_run = load_pretrained_encoder(
    project=ENCODER_PROJECT,
    run_id=ENCODER_RUN_ID,
    save_dir=SAVE_DIR,
    freeze=FREEZE_ENCODER,
    verbose=True,
)

N_LATENT = adapter.n_latent
print(f"Encoder type:    {type(adapter.encoder).__name__}")
print(f"n_latent:        {N_LATENT}")
print(f"context_margin:  {adapter.context_margin}")
print(f"Frozen:          {FREEZE_ENCODER}")

## 2. Data Hyperparameters (from encoder config)

In [ ]:
NUM_ICS = int(encoder_cfg.data.trajectory_params.num_ics)
N_PERIODS = int(encoder_cfg.data.trajectory_params.n_periods)
PTS_PER_PERIOD = int(encoder_cfg.data.trajectory_params.pts_per_period)
SEQ_LENGTH = int(encoder_cfg.data.train_test_params.seq_length)
OBS_NOISE = float(encoder_cfg.data.postprocessing.obs_noise)

delay_params = encoder_cfg.data.train_test_params.delay_embedding_params
OBSERVED_INDICES = list(delay_params.observed_indices) if delay_params.observed_indices != "all" else "all"
N_DELAYS = int(delay_params.get("n_delays", 1))
DELAY_SPACING = int(delay_params.get("delay_spacing", 1))

print(f"Data config (from encoder):")
print(f"  NUM_ICS={NUM_ICS}, N_PERIODS={N_PERIODS}, PTS_PER_PERIOD={PTS_PER_PERIOD}")
print(f"  SEQ_LENGTH={SEQ_LENGTH}, OBS_NOISE={OBS_NOISE}")
print(f"  OBSERVED_INDICES={OBSERVED_INDICES}, N_DELAYS={N_DELAYS}")

## 3. JacobianODE Hyperparameters (fixed across sweep)

In [ ]:
# ----------------------------------------------------------------
# JacobianODE integration
# ----------------------------------------------------------------
PREDICTION_STEPS = 10
TRAJ_INIT_STEPS = 15
INTERP_PTS = 4
INNER_N = 20
JAC_WINDOW_STRIDE = PREDICTION_STEPS  # non-overlapping

# ----------------------------------------------------------------
# Jacobian MLP architecture
# ----------------------------------------------------------------
JAC_HIDDEN_DIM = [256, 512, 512]
JAC_NUM_LAYERS = 3
JAC_ACTIVATION = 'silu'

# ----------------------------------------------------------------
# Fixed loss weights (loop_closure_weight is swept)
# ----------------------------------------------------------------
RECONSTRUCTION_LOSS_WEIGHT = 0.0 if FREEZE_ENCODER else 1.0
LATENT_PREDICTION_LOSS_WEIGHT = 0.0
JAC_CONSISTENCY_WEIGHT = 0.0
FNN_WEIGHT = 0.0 if FREEZE_ENCODER else 0.001

# ----------------------------------------------------------------
# Training
# ----------------------------------------------------------------
BATCH_SIZE = 16
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
MAX_EPOCHS = 200
LIMIT_TRAIN_BATCHES = 500
LIMIT_VAL_BATCHES = 50
ACCUMULATE_GRAD_BATCHES = 4
EARLY_STOPPING_PATIENCE = 5

## 4. Build Base Config

In [ ]:
_hidden_dim_str = str(JAC_HIDDEN_DIM).replace(' ', '')
_obs_idx_str = str(list(OBSERVED_INDICES)).replace(' ', '') if OBSERVED_INDICES != "all" else "all"

_enc = encoder_cfg.model.encoder

overrides = [
    # Model template
    "model=latent_ssm",
    # --- Encoder: propagate actual pretrained encoder architecture ---
    f"model.encoder.n_latent={N_LATENT}",
    f"model.encoder.d_model={int(_enc.get('d_model', 64))}",
    f"model.encoder.d_state={int(_enc.get('d_state', 64))}",
    f"model.encoder.n_layers={int(_enc.get('n_layers', 3))}",
    f"model.encoder.ffn_expand={int(_enc.get('ffn_expand', 2))}",
    f"model.encoder.r_min={float(_enc.get('r_min', 0.0))}",
    f"model.encoder.r_max={float(_enc.get('r_max', 0.99))}",
    f"model.encoder.dropout={float(_enc.get('dropout', 0.1))}",
    f"model.encoder.use_positional_encoding={bool(_enc.get('use_positional_encoding', True))}",
    f"model.encoder.positional_encoding_type={_enc.get('positional_encoding_type', 'sinusoidal')}",
    f"model.encoder.decoder_hidden={int(_enc.get('decoder_hidden', 128))}",
    f"model.encoder.decoder_layers={int(_enc.get('decoder_layers', 2))}",
    f"model.encoder.context_margin={adapter.context_margin}",
    f"model.prediction_steps={PREDICTION_STEPS}",
    f"model.params.hidden_dim={_hidden_dim_str}",
    f"model.params.num_layers={JAC_NUM_LAYERS}",
    f"model.params.activation={JAC_ACTIVATION}",
    
    # Data (match encoder)
    "data=dysts",
    "data.flow._target_=JacobianODE.dysts_sim.flows.Lorenz",
    f"data.trajectory_params.num_ics={NUM_ICS}",
    f"data.trajectory_params.n_periods={N_PERIODS}",
    f"data.trajectory_params.pts_per_period={PTS_PER_PERIOD}",
    f"data.postprocessing.obs_noise={OBS_NOISE}",
    f"data.train_test_params.seq_length={SEQ_LENGTH}",
    f"data.train_test_params.delay_embedding_params.observed_indices={_obs_idx_str}",
    f"data.train_test_params.delay_embedding_params.n_delays={N_DELAYS}",
    f"data.train_test_params.delay_embedding_params.delay_spacing={DELAY_SPACING}",
    
    # Training
    f"training.batch_size={BATCH_SIZE}",
    f"training.logger.save_dir={SAVE_DIR}",
    f"training.lightning.optimizer_kwargs.lr={LEARNING_RATE}",
    f"training.lightning.optimizer_kwargs.weight_decay={WEIGHT_DECAY}",
    f"training.lightning.reconstruction_loss_weight={RECONSTRUCTION_LOSS_WEIGHT}",
    f"training.lightning.latent_prediction_loss_weight={LATENT_PREDICTION_LOSS_WEIGHT}",
    f"training.lightning.jac_consistency_weight={JAC_CONSISTENCY_WEIGHT}",
    f"training.lightning.fnn_weight={FNN_WEIGHT}",
    "training.lightning.alpha_teacher_forcing=1",
    "training.lightning.teacher_forcing_annealing=True",
    "training.lightning.gamma_teacher_forcing=0.999",
    "training.lightning.loop_closure_training=True",
    "training.lightning.trajectory_training=True",
    f"training.lightning.jacobianODEint_kwargs.traj_init_steps={TRAJ_INIT_STEPS}",
    f"training.lightning.jacobianODEint_kwargs.interp_pts={INTERP_PTS}",
    f"training.lightning.jacobianODEint_kwargs.inner_N={INNER_N}",
    "training.lightning.jacobianODEint_kwargs.inner_path=line",
    f"training.trainer_params.max_epochs={MAX_EPOCHS}",
    f"training.trainer_params.limit_train_batches={LIMIT_TRAIN_BATCHES}",
    f"training.trainer_params.limit_val_batches={LIMIT_VAL_BATCHES}",
    f"training.trainer_params.accumulate_grad_batches={ACCUMULATE_GRAD_BATCHES}",
    f"training.early_stopping.early_stopping_patience={EARLY_STOPPING_PATIENCE}",
]

cfg = load_config(overrides=overrides)
cfg = initialize_config(cfg)

# Auto-generate project name
data_cls = cfg.data.flow._target_.split('.')[-1]
if WANDB_PROJECT is None:
    WANDB_PROJECT = f"{data_cls}__PretrainedEncoderJacODE"
WANDB_PROJECT_PATH = f"{WANDB_ENTITY}/{WANDB_PROJECT}"

print(f"W&B project: {WANDB_PROJECT_PATH}")
print(f"Jacobian MLP: input_dim={cfg.model.params.input_dim}, output_dim={cfg.model.params.output_dim}")

## 5. Generate Data (once, shared across sweep)

In [ ]:
seed_everything(cfg.data.flow.random_state)
eq, sol, dt = make_trajectories(cfg, verbose=True)
print(f"\nFull trajectory shape: {sol['values'].shape}")
print(f"Time step dt = {dt:.4f}")

In [ ]:
result = postprocess_data(cfg, sol["values"])
values = result.values
mu = result.mu
sigma = result.sigma
noise_scale_factor = result.noise_scale_factor

cfg.data.postprocessing.mu = float(mu)
cfg.data.postprocessing.sigma = float(sigma)
cfg.data.postprocessing.noise_scale_factor = float(noise_scale_factor)

train_dl, val_dl, test_dl, trajs = create_dataloaders(cfg, values, verbose=True, return_full_obs=True)
n_obs = trajs['train_trajs'].sequence.shape[-1]
n_dims = values.shape[-1]

print(f"\nn_obs = {n_obs}, n_dims = {n_dims}")
print(f"sqrt(n_dims) = {np.sqrt(n_dims):.4f} (C2 loop-closure threshold)")

## 6. Launch Sweep over `loop_closure_weight`

### Option A: SLURM multirun (for cluster)

Build a Hydra `--multirun` command and submit to SLURM. **Note**: the SLURM sweep
uses Hydra's config, not the pre-trained adapter directly. You'll need a custom
entry point or use **Option B** (local loop) instead.

### Option B: Local loop (recommended)

Trains each run in sequence in this notebook. Uses the pre-trained adapter directly.

In [ ]:
# ----------------------------------------------------------------
# Check which lambda values already have completed runs on W&B
# ----------------------------------------------------------------
api = wandb.Api()
try:
    existing_runs = api.runs(WANDB_PROJECT_PATH)
    print(f"Found {len(existing_runs)} total runs in {WANDB_PROJECT_PATH}")
except Exception as e:
    print(f"Could not query project (may not exist yet): {e}")
    existing_runs = []

already_run = []
remaining_lambdas = []
for lam in LAMBDA_LOOP_VALUES:
    matching = [
        run for run in existing_runs
        if 'training' in run.config
        and 'lightning' in run.config['training']
        and abs(run.config['training']['lightning'].get('loop_closure_weight', -1) - lam) < 1e-10
        and run.state == 'finished'
    ]
    if matching:
        already_run.append(lam)
    else:
        remaining_lambdas.append(lam)

print(f"\nAlready completed: {already_run}")
print(f"Remaining to run:  {remaining_lambdas}")

In [ ]:
# ----------------------------------------------------------------
# Option B: Local loop — train each lambda value sequentially
# ----------------------------------------------------------------
import copy

TRUE_LYAPUNOV = [0.91, 0.0, -14.57]
freeze_tag = "frozen" if FREEZE_ENCODER else "unfrozen"
encoder_type = type(adapter.encoder).__name__

for lam in tqdm(remaining_lambdas, desc="Sweep"):
    print(f"\n{'='*60}")
    print(f"Training with loop_closure_weight = {lam}")
    print(f"{'='*60}")
    
    # Deep copy config and override loop closure weight
    sweep_cfg = copy.deepcopy(cfg)
    sweep_cfg.training.lightning.loop_closure_weight = lam
    
    # Fresh Jacobian MLP for each run
    seed_everything(cfg.data.flow.random_state + 1)
    jac_model = instantiate(sweep_cfg.model.params)
    
    # Clone adapter if unfrozen (each run gets fresh encoder copy)
    # If frozen, reuse the same adapter (weights won't change)
    run_adapter = adapter if FREEZE_ENCODER else adapter.clone()
    
    # Build model
    lit_model = instantiate(
        sweep_cfg.training.lightning,
        model=jac_model,
        encoder=run_adapter,
        dt=dt,
        save_dir=SAVE_DIR,
        mu=float(mu),
        sigma=float(sigma),
        noise_scale_factor=float(noise_scale_factor),
        prediction_steps=PREDICTION_STEPS,
    )
    lit_model.true_lyapunov_exponents = torch.tensor(TRUE_LYAPUNOV)
    
    # Run name
    name = (
        f"{data_cls}__{encoder_type}__n{N_LATENT}__{freeze_tag}"
        f"__lc{lam}__pred{PREDICTION_STEPS}"
    )
    
    # Clean up stale W&B
    try:
        wandb.finish(quiet=True)
    except Exception:
        pass
    
    train_model(
        cfg=sweep_cfg,
        lit_model=lit_model,
        train_dataloaders=train_dl,
        val_dataloaders=val_dl,
        name=name,
        project=WANDB_PROJECT,
        entity=WANDB_ENTITY,
    )

print("\nSweep complete!")

## 7. Collect W&B Runs and Model Selection

### 7a. Collect Run IDs

In [ ]:
api = wandb.Api()
all_runs = api.runs(WANDB_PROJECT_PATH)
print(f"Found {len(all_runs)} total runs in {WANDB_PROJECT_PATH}")

# Filter to finished runs matching our encoder type
sweep_run_ids = []
sweep_lambdas = []

for run in all_runs:
    if run.state != 'finished':
        continue
    lc_weight = run.config.get('training', {}).get('lightning', {}).get('loop_closure_weight')
    if lc_weight is not None and lc_weight in LAMBDA_LOOP_VALUES:
        sweep_run_ids.append(run.id)
        sweep_lambdas.append(lc_weight)

# Sort by lambda value
sorted_pairs = sorted(zip(sweep_lambdas, sweep_run_ids))
sweep_lambdas = [p[0] for p in sorted_pairs]
sweep_run_ids = [p[1] for p in sorted_pairs]

print(f"\nFound {len(sweep_run_ids)} matching runs:")
for lam, rid in zip(sweep_lambdas, sweep_run_ids):
    print(f"  lambda={lam} -> run_id={rid}")

### 7b. Compute Diagnostics and Select Best Model

In [ ]:
# Use the tuning module's select_from_wandb_runs for physics-informed selection
# This loads each model, computes C1/C2/C3, and returns the best
sweep_result = select_from_wandb_runs(
    run_ids=sweep_run_ids,
    project=WANDB_PROJECT_PATH,
    dt=dt,
    n_dims=n_dims,
    eigenvalue_threshold=0.001,
    use_loop_closure=True,
    lambda_values=sweep_lambdas,
    verbose=True,
)

result = sweep_result.selection
all_diagnostics = sweep_result.diagnostics

print("\n" + "=" * 60)
print("MODEL SELECTION RESULT")
print("=" * 60)
print(f"Best lambda:    {sweep_lambdas[result.best_index]}")
print(f"Best run ID:    {sweep_run_ids[result.best_index]}")
print(f"Best traj loss: {result.best_metrics.trajectory_val_loss:.6f}")
print(f"\nCriteria applied (after relaxation): {result.criteria_applied}")
print(f"Surviving models: {len(result.surviving_indices)} / {len(all_diagnostics)}")

### 7c. Visualize Selection

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

one_step_mases = [m.one_step_mase for m in all_diagnostics]
loop_closure_losses = [m.loop_closure_loss for m in all_diagnostics]
eig_fracs = [m.fast_eigenvalue_fraction for m in all_diagnostics]
traj_losses = [m.trajectory_val_loss for m in all_diagnostics]

colors = ['tab:green' if i in result.surviving_indices else 'tab:red'
          for i in range(len(all_diagnostics))]
x_labels = [str(v) for v in sweep_lambdas]
x_pos = range(len(sweep_lambdas))

# Panel 1: One-step MASE
axes[0, 0].bar(x_pos, one_step_mases, color=colors)
axes[0, 0].axhline(y=1.0, color='k', linestyle='--', lw=1, label='persistence')
axes[0, 0].set_xticks(x_pos); axes[0, 0].set_xticklabels(x_labels)
axes[0, 0].set_ylabel('MASE'); axes[0, 0].set_title('C1: One-step MASE')
axes[0, 0].legend()

# Panel 2: Loop closure loss
axes[0, 1].bar(x_pos, loop_closure_losses, color=colors)
axes[0, 1].axhline(y=np.sqrt(n_dims), color='k', linestyle='--', lw=1, label=f'sqrt(n_dims)={np.sqrt(n_dims):.2f}')
axes[0, 1].set_xticks(x_pos); axes[0, 1].set_xticklabels(x_labels)
axes[0, 1].set_ylabel('Loop closure loss'); axes[0, 1].set_title('C2: Loop Closure')
axes[0, 1].legend()

# Panel 3: Fast eigenvalue fraction
axes[1, 0].bar(x_pos, eig_fracs, color=colors)
axes[1, 0].set_xticks(x_pos); axes[1, 0].set_xticklabels(x_labels)
axes[1, 0].set_ylabel('Fraction fast eigenvalues'); axes[1, 0].set_title('C3: Eigenvalue Fraction')

# Panel 4: Trajectory val loss
axes[1, 1].bar(x_pos, traj_losses, color=colors)
axes[1, 1].set_xticks(x_pos); axes[1, 1].set_xticklabels(x_labels)
axes[1, 1].set_ylabel('Trajectory val loss'); axes[1, 1].set_title('Trajectory Loss (lower = better)')

for ax in axes.flat:
    ax.set_xlabel('loop_closure_weight')

fig.suptitle('Loop Closure Weight Sweep — Model Selection', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 8. Load the Best Model for Downstream Use

In [ ]:
best_run_id = sweep_run_ids[result.best_index]
best_lambda = sweep_lambdas[result.best_index]

print(f"Loading best model: run_id={best_run_id}, lambda={best_lambda}")

run, best_cfg, eq, dt, values, _, _, _, _, best_lit_model = load_run(
    WANDB_PROJECT_PATH,
    run_id=best_run_id,
    save_dir=SAVE_DIR,
    generate_data=True,
    verbose=True,
)

# Rebuild dataloaders from the loaded data
train_dl, val_dl, test_dl, trajs = create_dataloaders(best_cfg, values, verbose=True, return_full_obs=True)

print(f"\nModel type: {type(best_lit_model).__name__}")
print(f"Total params: {sum(p.numel() for p in best_lit_model.parameters()):,}")

### 8a. Lyapunov Exponent Comparison (Best Model)

In [ ]:
TRUE_LYAPUNOV = [0.91, 0.0, -14.57]

best_lit_model.eval()
with torch.no_grad():
    test_batch = next(iter(test_dl))
    if isinstance(test_batch, (list, tuple)):
        test_batch = test_batch[0]
    z_test = best_lit_model.encode_trajectory(test_batch)
    z_long = z_test[:5].reshape(-1, z_test.shape[-1]).unsqueeze(0)
    jacs_long = best_lit_model.compute_jacobians(z_long)[0]
    pred_lyap = LitLatentJacobianODE.compute_lyapunov_exponents(jacs_long, dt)

print("Predicted Lyapunov exponents (best model):")
for i, le in enumerate(pred_lyap):
    print(f"  lambda_{i+1} = {le.item():+.4f}")
print(f"\nTrue Lorenz:")
for i, le in enumerate(TRUE_LYAPUNOV):
    print(f"  lambda_{i+1} = {le:+.4f}")

fig, ax = plt.subplots(figsize=(8, 4))
pred_np = pred_lyap.numpy()
ax.bar(np.arange(len(pred_np)) - 0.15, pred_np, width=0.3, label="Predicted", alpha=0.8)
ax.bar(np.arange(len(TRUE_LYAPUNOV)) + 0.15, TRUE_LYAPUNOV, width=0.3, label="True", alpha=0.8)
ax.axhline(y=0, color='k', linestyle='--', lw=0.5)
ax.set_xlabel("Exponent index")
ax.set_ylabel("Lyapunov exponent")
ax.set_title(f"Lyapunov Spectrum (best lambda={best_lambda})")
ax.legend()
plt.tight_layout()
plt.show()